# Financial Analysis Chatbot

## Objective

Develop a simple financial chatbot that uses the financial data analyzed in Task 1 to answer predefined questions and provide clear, easy-to-understand financial insights.

The chatbot covers **Microsoft, Tesla, and Apple** and retrieves information directly from the provided CSV datasets.

## Features

- Data retrieval from CSV files
- Total revenue, net income, profit margin, assets, liabilities, and operating cash flow
- Year-over-year comparisons
- Company comparisons
- Simple natural-language query matching
- Clear, jargon-free responses
- Suggested follow-up topics
- Demonstration and testing of predefined queries

> **Note:** This is a rule-based financial chatbot using predefined retrieval logic. It does not use an external generative AI API.

## Step 1: Preparation

The financial datasets created during Task 1 are used as the chatbot's knowledge base.

Files used:
- `microsoft_financial_data.csv`
- `tesla_financial_data.csv`
- `apple_financial_data.csv`

The chatbot combines these files into one structured DataFrame so that financial values can be retrieved easily.

In [1]:
import pandas as pd

# Load the financial datasets
microsoft = pd.read_csv("microsoft_financial_data.csv")
tesla = pd.read_csv("tesla_financial_data.csv")
apple = pd.read_csv("apple_financial_data.csv")

# Combine all company data
df = pd.concat([microsoft, tesla, apple], ignore_index=True)

# Calculate net profit margin
df["Net Profit Margin (%)"] = (
    df["Net Income"] / df["Total Revenue"]
) * 100

# Sort the data for easier retrieval
df = df.sort_values(["Company", "Year"]).reset_index(drop=True)

print("Financial data loaded successfully.")
print(f"Companies: {', '.join(df['Company'].unique())}")
print(f"Years: {df['Year'].min()}-{df['Year'].max()}")
df

Financial data loaded successfully.
Companies: Apple, Microsoft, Tesla
Years: 2023-2025


,Company,Year,Total Revenue,Net Income,Total Assets,Total Liabilities,Cash Flow from Operating Activities,Net Profit Margin (%)
0,Apple,2023,383285,96995,352583,290437,110543,25.306234
1,Apple,2024,391035,93736,364980,308030,118254,23.971256
2,Apple,2025,416161,112010,359241,285508,111482,26.915064
3,Microsoft,2023,211915,72361,411976,205753,87582,34.146238
4,Microsoft,2024,245122,88136,512163,243686,118548,35.955973
5,Microsoft,2025,281724,101832,619003,275524,136162,36.146015
6,Tesla,2023,96773,14974,106618,43009,13256,15.473324
7,Tesla,2024,97690,7153,122070,48390,14923,7.322141
8,Tesla,2025,94827,3855,137806,54941,14747,4.065298


## Step 2: Data Structuring

The chatbot uses the following financial metrics:

- Total Revenue
- Net Income
- Net Profit Margin
- Total Assets
- Total Liabilities
- Cash Flow from Operating Activities

All monetary values in the source data are in **USD millions**.

In [2]:
# Quick structured view
metrics = [
    "Total Revenue",
    "Net Income",
    "Net Profit Margin (%)",
    "Total Assets",
    "Total Liabilities",
    "Cash Flow from Operating Activities"
]

df[["Company", "Year"] + metrics]

,Company,Year,Total Revenue,Net Income,Net Profit Margin (%),Total Assets,Total Liabilities,Cash Flow from Operating Activities
0,Apple,2023,383285,96995,25.306234,352583,290437,110543
1,Apple,2024,391035,93736,23.971256,364980,308030,118254
2,Apple,2025,416161,112010,26.915064,359241,285508,111482
3,Microsoft,2023,211915,72361,34.146238,411976,205753,87582
4,Microsoft,2024,245122,88136,35.955973,512163,243686,118548
5,Microsoft,2025,281724,101832,36.146015,619003,275524,136162
6,Tesla,2023,96773,14974,15.473324,106618,43009,13256
7,Tesla,2024,97690,7153,7.322141,122070,48390,14923
8,Tesla,2025,94827,3855,4.065298,137806,54941,14747


## Step 3: Retrieval Methods

The following helper functions retrieve the correct financial record based on company and year.

This keeps the chatbot logic simple and makes it easy to map a user question to a specific data point.

In [8]:
def get_company_data(company, year=None):
    company = company.title()

    result = df[df["Company"] == company]

    if year is not None:
        result = result[result["Year"] == int(year)]

    return result


def get_latest_data(company):
    result = get_company_data(company)

    if result.empty:
        return None

    return result.sort_values("Year").iloc[-1]


def format_usd_millions(value):
    return f"USD {value / 1000:.1f}B"


def format_percent(value):
    return f"{value:.2f}%"


## Step 4: Financial Response Functions

These functions generate the chatbot's answers from the actual financial dataset rather than using hard-coded financial values.

In [9]:
def company_overview(company):
    row = get_latest_data(company)

    if row is None:
        return f"I could not find data for {company}."

    return (
        f"{company.title()} in {int(row['Year'])}: "
        f"revenue was {format_usd_millions(row['Total Revenue'])}, "
        f"net income was {format_usd_millions(row['Net Income'])}, "
        f"net profit margin was {format_percent(row['Net Profit Margin (%)'])}, "
        f"total assets were {format_usd_millions(row['Total Assets'])}, "
        f"and operating cash flow was "
        f"{format_usd_millions(row['Cash Flow from Operating Activities'])}."
    )


def metric_response(company, metric, year=None):
    row = get_company_data(company, year)

    if row.empty:
        return f"I could not find data for {company}."

    row = row.sort_values("Year").iloc[-1]

    value = row[metric]

    if metric == "Net Profit Margin (%)":
        formatted = format_percent(value)
    else:
        formatted = format_usd_millions(value)

    return (
        f"{company.title()}'s {metric.lower()} in {int(row['Year'])} "
        f"was {formatted}."
    )


def yearly_change(company, metric, year1=2024, year2=2025):
    company_data = get_company_data(company)

    row1 = company_data[company_data["Year"] == year1]
    row2 = company_data[company_data["Year"] == year2]

    if row1.empty or row2.empty:
        return f"I do not have enough data to compare {company} for those years."

    value1 = row1.iloc[0][metric]
    value2 = row2.iloc[0][metric]

    change = value2 - value1
    pct_change = (change / value1) * 100

    direction = "increased" if change >= 0 else "decreased"

    if metric == "Net Profit Margin (%)":
        return (
            f"{company.title()}'s {metric.lower()} {direction} from "
            f"{format_percent(value1)} in {year1} to {format_percent(value2)} "
            f"in {year2}, a change of {pct_change:.2f}%."
        )

    return (
        f"{company.title()}'s {metric.lower()} {direction} from "
        f"{format_usd_millions(value1)} in {year1} to "
        f"{format_usd_millions(value2)} in {year2}, "
        f"a change of {pct_change:.2f}%."
    )


def compare_revenue(year=2025):
    data = df[df["Year"] == year].sort_values("Total Revenue", ascending=False)

    parts = [
        f"{row['Company']}: {format_usd_millions(row['Total Revenue'])}"
        for _, row in data.iterrows()
    ]

    return f"Revenue comparison for {year}: " + "; ".join(parts) + "."


def compare_profit_margin(year=2025):
    data = df[df["Year"] == year].sort_values(
        "Net Profit Margin (%)", ascending=False
    )

    parts = [
        f"{row['Company']}: {format_percent(row['Net Profit Margin (%)'])}"
        for _, row in data.iterrows()
    ]

    return f"Net profit margin comparison for {year}: " + "; ".join(parts) + "."


def company_trend(company):
    data = get_company_data(company)

    if data.empty:
        return f"I could not find data for {company}."

    first = data.iloc[0]
    last = data.iloc[-1]

    revenue_change = (
        (last["Total Revenue"] - first["Total Revenue"])
        / first["Total Revenue"]
    ) * 100

    income_change = (
        (last["Net Income"] - first["Net Income"])
        / first["Net Income"]
    ) * 100

    return (
        f"From {int(first['Year'])} to {int(last['Year'])}, "
        f"{company.title()}'s revenue changed by {revenue_change:.2f}% "
        f"and net income changed by {income_change:.2f}%."
    )


## Step 5: Chatbot Logic

The chatbot recognizes common financial questions and maps them to the appropriate retrieval function.

Supported examples include:

1. Company overview
2. Total revenue
3. Net income
4. Net profit margin
5. Total assets
6. Total liabilities
7. Operating cash flow
8. Year-over-year changes
9. Revenue comparison
10. Profit-margin comparison
11. Overall company trend

In [10]:
def simple_chatbot(user_query):
    query = user_query.lower().strip()

    companies = ["microsoft", "tesla", "apple"]
    company = next((c for c in companies if c in query), None)

    # Help / supported questions
    if query in ["help", "hi", "hello", "what can you do"]:
        return (
            "I can answer questions about Microsoft, Tesla, and Apple. "
            "Try asking about revenue, net income, profit margin, assets, "
            "liabilities, operating cash flow, yearly changes, or company comparisons."
        )

    # Company overview
    if company and any(word in query for word in [
        "overview", "summary", "performance", "financial position"
    ]):
        return company_overview(company)

    # Revenue comparison
    if "compare" in query and "revenue" in query:
        return compare_revenue()

    # Profit margin comparison
    if "compare" in query and (
        "profit margin" in query or "profitability" in query
    ):
        return compare_profit_margin()

    # General trend
    if company and any(word in query for word in [
        "trend", "over the years", "last three years", "2023 to 2025"
    ]):
        return company_trend(company)

    # Year-over-year changes
    if company and any(word in query for word in [
        "changed", "change", "growth", "increased", "decreased", "last year"
    ]):
        if "revenue" in query:
            return yearly_change(company, "Total Revenue")
        if "net income" in query or "income" in query:
            return yearly_change(company, "Net Income")
        if "cash flow" in query:
            return yearly_change(
                company,
                "Cash Flow from Operating Activities"
            )
        if "profit margin" in query:
            return yearly_change(company, "Net Profit Margin (%)")

    # Specific financial metrics
    if company:
        if "revenue" in query:
            return metric_response(company, "Total Revenue")

        if "net income" in query or "income" in query:
            return metric_response(company, "Net Income")

        if "profit margin" in query or "profitability" in query:
            return metric_response(company, "Net Profit Margin (%)")

        if "assets" in query:
            return metric_response(company, "Total Assets")

        if "liabilities" in query:
            return metric_response(company, "Total Liabilities")

        if "cash flow" in query or "operating cash" in query:
            return metric_response(
                company,
                "Cash Flow from Operating Activities"
            )

    return (
        "Sorry, I can only answer predefined financial questions. "
        "Try: 'What is Microsoft's revenue?', "
        "'How has Tesla's net income changed?', "
        "'Compare revenue', or 'What is Apple's profit margin?'"
    )


## Step 6: Testing the Chatbot

The following predefined queries test whether the chatbot can retrieve and communicate financial insights correctly.

In [11]:
test_queries = [
    "What is Microsoft's revenue?",
    "What is Apple's net income?",
    "What is Tesla's profit margin?",
    "How has Microsoft's revenue changed?",
    "How has Tesla's net income changed?",
    "What is Apple's operating cash flow?",
    "Compare revenue",
    "Compare profit margin",
    "Give me Microsoft's overview",
    "Show Apple's trend"
]

for question in test_queries:
    print("User:", question)
    print("Bot:", simple_chatbot(question))
    print("-" * 90)


User: What is Microsoft's revenue?
Bot: Microsoft's total revenue in 2025 was USD 281.7B.
------------------------------------------------------------------------------------------
User: What is Apple's net income?
Bot: Apple's net income in 2025 was USD 112.0B.
------------------------------------------------------------------------------------------
User: What is Tesla's profit margin?
Bot: Tesla's net profit margin (%) in 2025 was 4.07%.
------------------------------------------------------------------------------------------
User: How has Microsoft's revenue changed?
Bot: Microsoft's total revenue increased from USD 245.1B in 2024 to USD 281.7B in 2025, a change of 14.93%.
------------------------------------------------------------------------------------------
User: How has Tesla's net income changed?
Bot: Tesla's net income decreased from USD 7.2B in 2024 to USD 3.9B in 2025, a change of -46.11%.
----------------------------------------------------------------------------------

## Step 7: Interactive Chatbot

Run the cell below to interact with the chatbot using the keyboard.

Type `help` to see supported topics and type `exit` to end the conversation.

In [12]:
print("Financial Analysis Chatbot")
print("Type 'help' for examples or 'exit' to quit.\n")

while True:
    user_query = input("You: ")

    if user_query.lower().strip() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye! Thanks for using the Financial Analysis Chatbot.")
        break

    print("Bot:", simple_chatbot(user_query))
    print()


Financial Analysis Chatbot
Type 'help' for examples or 'exit' to quit.



You:  What is Microsoft's revenue?


Bot: Microsoft's total revenue in 2025 was USD 281.7B.



You:  Compare revenue


Bot: Revenue comparison for 2025: Apple: USD 416.2B; Microsoft: USD 281.7B; Tesla: USD 94.8B.



You:  exit


Bot: Goodbye! Thanks for using the Financial Analysis Chatbot.


## Step 8: Demonstration and Documentation

### How the chatbot works

1. The chatbot loads the three financial CSV files.
2. The files are combined into a single structured dataset.
3. Net profit margin is calculated from net income and total revenue.
4. The user's question is checked for a company and financial topic.
5. The appropriate retrieval function finds the relevant data.
6. The chatbot formats the result into simple, readable language.
7. For unsupported questions, the chatbot explains which predefined topics are available.

### Example queries

- What is Microsoft's revenue?
- What is Apple's net income?
- What is Tesla's profit margin?
- How has Microsoft's revenue changed?
- How has Tesla's net income changed?
- What is Apple's operating cash flow?
- Compare revenue
- Compare profit margin
- Give me Microsoft's overview
- Show Apple's trend

### Visual aids

The financial charts created in **Task 1** can be used alongside this chatbot to visually explain revenue, net income, and operating cash-flow trends. The chatbot itself focuses on text-based interaction.

### Limitations

This chatbot uses predefined retrieval rules rather than a large language model. Therefore, it can answer supported financial questions but may not understand completely new or highly complex questions. The accuracy of its responses depends on the financial data provided in the CSV files.